# Pipeline EHBG-FACS · 02 · Metaheurísticas (ACO y Tabu)

**Paradigma 2 — implementaciones oficiales de SVRPBench, re-puntuadas con CRN.**

Envuelve el **Ant System** y la **Tabu Search** oficiales del benchmark (con arranque NN+2opt) y re-puntúa sus rutas con el evaluador compartido, para que sean comparables con el resto. Agregación best-of-K (multistart) determinista por instancia.

In [ ]:
# === Configuración del entorno (ejecuta esta celda primero) =================
# Requiere: (a) el paquete `svrplab` (carpeta experiments/colab del repo de tesis)
#           (b) el repo oficial de SVRPBench (se clona solo en bootstrap.init()).
REPO_URL  = "https://github.com/AbrahaHub/TESIS-ANT"   # <-- EDITA si tu repo difiere
USE_DRIVE = True   # persistir banco/resultados/modelos en Google Drive (recomendado)

import os, sys, subprocess

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no disponible (¿ejecutas local?):", e)

def _find_svrplab():
    cands = ["/content/drive/MyDrive/TESIS-ANT/experiments/colab",
             "/content/TESIS-ANT/experiments/colab",
             os.path.join(os.getcwd(), "experiments", "colab"),
             os.getcwd()]
    for c in cands:
        if os.path.isdir(os.path.join(c, "svrplab")):
            return c
    return None

_path = _find_svrplab()
if _path is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TESIS-ANT"], check=False)
    _path = "/content/TESIS-ANT/experiments/colab"
sys.path.insert(0, _path)
print("svrplab en:", _path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas",
                "matplotlib", "scikit-learn", "pillow", "tqdm"], check=False)
# torch ya viene en Colab. gurobipy solo se instala en el notebook del paradigma 1.

from svrplab import bootstrap, protocol, data, runner, metrics, viz
env   = bootstrap.init()        # GPU + repo oficial SVRPBench + rutas (Drive si está montado)
proto = protocol.DEFAULT
print("device:", env.device, "| raíz de artefactos:", env.paths.root)

In [ ]:
# === Configuración del experimento (IDÉNTICA en los 5 notebooks) ============
# Para garantizar el "piso parejo", TODOS los notebooks deben usar los MISMOS
# SIZES y N_INSTANCES: así resuelven exactamente el mismo banco de instancias.
SIZES       = [10, 20, 50]           # clientes. Extiende a [50,100,200,300] (ver notas).
N_INSTANCES = proto.instances_per_size   # 30 (rigor estadístico). Corrida rápida: pon 5.

bank = data.load_bank(env.paths.instances, SIZES, N_INSTANCES,
                      base_seed=proto.base_seed, capacity_mode=proto.capacity_mode, verbose=True)
print({s: len(v) for s, v in bank.items()}, "instancias por tamaño")

## Resolver ACO y Tabu

In [ ]:
from svrplab.solvers.metaheuristic import ACO, Tabu
import pandas as pd
df_aco  = runner.run_solver(ACO(n_seeds=5),  "aco",  bank, env, proto, verbose=True)
df_tabu = runner.run_solver(Tabu(n_seeds=5), "tabu", bank, env, proto, verbose=True)
df = pd.concat([df_aco, df_tabu], ignore_index=True)
df

## Métricas y figuras

In [ ]:
display(metrics.aggregate_by_size(df))
import matplotlib.pyplot as plt
viz.plot_comparison(df); plt.show()
inst = bank[SIZES[0]][0]
sol = ACO(n_seeds=5).solve(inst, num_realizations=proto.realizations)
viz.plot_routes(inst, sol.routes, title=f"aco · n={SIZES[0]}"); plt.show()

**Interpretación.** Las metaheurísticas alcanzan **factibilidad alta** pero usando **más vehículos** (rutas cortas) y, por tanto, mayor costo: la otra familia del tradeoff.